In [1]:
# simulate_allocation.ipynb

import random
import time
import numpy as np
import pandas as pd
from SSELF_base import FootprintDatabase
from allocation_extension import AllocationSystem, AllocationCompany, AllocationProduct

# Step 1: Create a system with multiple interconnected companies
num_companies = 10
base_products = num_companies
system = AllocationSystem(num_companies=num_companies, num_products=base_products)

# Step 2: Add hierarchies to one company and define two co-products using mass allocation
company = system.companies['Company_1']

sub_sales = pd.DataFrame.from_dict({
    system.product_id_counter: 300.0,  # Product A ($300)
    system.product_id_counter + 1: 200.0  # Product B ($200)
}, orient="index", columns=["Sales"])
sub_purchases = pd.Series({i + 1: np.random.uniform(50, 150) for i in range(3)})
direct_impacts = pd.DataFrame({"kg CO2eq": [random.randint(50, 150)]})

sub = AllocationCompany("Company_1_Sub_1", 2024, sub_purchases, sub_sales, direct_impacts)

product_A = AllocationProduct(system.product_id_counter, "Copper coil", "kg", sub, properties={"mass": 600})
product_B = AllocationProduct(system.product_id_counter + 1, "Copper residue", "kg", sub, properties={"mass": 400})
sub.add_product(product_A)
sub.add_product(product_B)
company.add_sub_company(sub)
system.product_id_counter += 2

# Step 3: Initialize footprint database
fp_db = FootprintDatabase(2024)
for c in system.companies.values():
    for p in c.products:
        fp_db.report(p.product_id, 0)
    for sub in getattr(c, "sub_companies", []):
        for p in sub.products:
            fp_db.report(p.product_id, 0)

# Step 4: Run iterative simulation
updates_completed = False
iteration = 0
start_time = time.time()

while not updates_completed:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")
    any_updated = False

    for name, company in system.companies.items():
        previous = company.latest_update

        for sub in getattr(company, "sub_companies", []):
            sub.update_footprint(fp_db)
            sub.report_footprint(fp_db)

        company.update_footprint(fp_db)
        new = company.latest_update

        if previous is None or not np.isclose(previous, new, atol=1e-6):
            print(f"  → {name} updated.")
            any_updated = True
            company.report_footprint(fp_db)

    updates_completed = not any_updated

end_time = time.time()
print("\n Updates completed.")
print(f" Time taken: {end_time - start_time:.2f} seconds")

print("\n Final Footprints:")
for name, company in system.companies.items():
    print(f"{name}: {company.latest_update:.4f} kg CO2e/unit")
    for sub in getattr(company, "sub_companies", []):
        print(f"  {sub.name}: {sub.latest_update:.4f}")

print("\n🗃 Final footprint database:")
print(fp_db.data.sort_values("id"))



--- Iteration 1 ---

Calculating footprint for Company_1_Sub_1
  Using mass allocation
  Product Copper coil (Mass=600): Assigned footprint 52.2000
  Product Copper residue (Mass=400): Assigned footprint 34.8000
  Final footprint for Company_1_Sub_1: 45.2400

Calculating footprint for Company_1
  Using economic allocation
  Product Product 1 (Sales=15.391540666077715): Assigned footprint 21.3754
  Final footprint for Company_1: 21.3754
  → Company_1 updated.

Calculating footprint for Company_2
  Using economic allocation
  Product Product 2 (Sales=12.860474650354979): Assigned footprint 42.8736
  Final footprint for Company_2: 42.8736
  → Company_2 updated.

Calculating footprint for Company_3
  Using economic allocation
  Product Product 3 (Sales=13.957392825756656): Assigned footprint 39.8534
  Final footprint for Company_3: 39.8534
  → Company_3 updated.

Calculating footprint for Company_4
  Using economic allocation
  Product Product 4 (Sales=11.6920009415796): Assigned footprin